In [0]:
import sys
# Clear stale matplotlib module cache (can become partially
# initialized on serverless compute, causing AttributeError)
for _mod in list(sys.modules):
    if _mod.startswith("matplotlib"):
        del sys.modules[_mod]
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix


In [0]:
train = pd.read_csv("/Volumes/workspace/default/disease_prediction/Training.csv")

test = pd.read_csv("/Volumes/workspace/default/disease_prediction/Testing.csv")

In [0]:
# Check Missing Values
train.isnull().sum()


itching                    0
skin_rash                  0
nodal_skin_eruptions       0
continuous_sneezing        0
shivering                  0
                        ... 
blister                    0
red_sore_around_nose       0
yellow_crust_ooze          0
prognosis                  0
Unnamed: 133            4920
Length: 134, dtype: int64

In [0]:
test.isnull().sum()

itching                 0
skin_rash               0
nodal_skin_eruptions    0
continuous_sneezing     0
shivering               0
                       ..
inflammatory_nails      0
blister                 0
red_sore_around_nose    0
yellow_crust_ooze       0
prognosis               0
Length: 133, dtype: int64

In [0]:
train = train.drop(columns=["Unnamed: 133"], errors="ignore")


In [0]:
train.dtypes

itching                  int64
skin_rash                int64
nodal_skin_eruptions     int64
continuous_sneezing      int64
shivering                int64
                         ...  
inflammatory_nails       int64
blister                  int64
red_sore_around_nose     int64
yellow_crust_ooze        int64
prognosis               object
Length: 133, dtype: object

In [0]:
test.dtypes

itching                  int64
skin_rash                int64
nodal_skin_eruptions     int64
continuous_sneezing      int64
shivering                int64
                         ...  
inflammatory_nails       int64
blister                  int64
red_sore_around_nose     int64
yellow_crust_ooze        int64
prognosis               object
Length: 133, dtype: object

In [0]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4920 entries, 0 to 4919
Columns: 133 entries, itching to prognosis
dtypes: int64(132), object(1)
memory usage: 5.0+ MB


In [0]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Columns: 133 entries, itching to prognosis
dtypes: int64(132), object(1)
memory usage: 43.8+ KB


In [0]:
print("Total Missing Values train:", train.isnull().sum().sum())

Total Missing Values train: 0


In [0]:
print("Total Missing Values test:", test.isnull().sum().sum())

Total Missing Values test: 0


In [0]:
print("Duplicate Rows train:", train .duplicated().sum())

Duplicate Rows train: 4616


In [0]:
print("Duplicate Rows test:", test.duplicated().sum())

Duplicate Rows test: 0


In [0]:
print("Total rows:", len(train))
print("Exact duplicates:", train.duplicated().sum())
print("Unique complete rows:", len(train.drop_duplicates()))

X = train.drop("prognosis", axis=1)

print("Duplicate symptom patterns:", X.duplicated().sum())

duplicate_symptoms = (
    train.groupby(list(X.columns))["prognosis"]
       .nunique()
)

print(
    "Conflicting symptom patterns:",
    (duplicate_symptoms > 1).sum()
)

Total rows: 4920
Exact duplicates: 4616
Unique complete rows: 304
Duplicate symptom patterns: 4616
Conflicting symptom patterns: 0


In [0]:
train_clean = train.drop_duplicates().copy()

print("Original rows:", len(train))
print("After removing duplicates:", len(train_clean))

Original rows: 4920
After removing duplicates: 304


In [0]:
print(
    train_clean["prognosis"].value_counts().sort_index()
)

prognosis
(vertigo) Paroymsal  Positional Vertigo     7
AIDS                                        5
Acne                                        5
Alcoholic hepatitis                         8
Allergy                                     5
Arthritis                                   6
Bronchial Asthma                            7
Cervical spondylosis                        6
Chicken pox                                10
Chronic cholestasis                         8
Common Cold                                 9
Dengue                                     10
Diabetes                                    9
Dimorphic hemmorhoids(piles)                6
Drug Reaction                               6
Fungal infection                            5
GERD                                        7
Gastroenteritis                             5
Heart attack                                5
Hepatitis B                                 9
Hepatitis C                                 7
Hepatitis D             

In [0]:
print(
    "Number of diseases:",
    train_clean["prognosis"].nunique()
)

Number of diseases: 41


In [0]:
X = train_clean.drop("prognosis", axis=1)
y = train_clean["prognosis"]
X = X.fillna(0)

In [0]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (304, 132)
y shape: (304,)


In [0]:
print(X.dtypes.value_counts())

int64    132
Name: count, dtype: int64


In [0]:
print("X shape:", X.shape)
print("Missing values:", X.isna().sum().sum())

X shape: (304, 132)
Missing values: 0


In [0]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print("Number of classes:", len(encoder.classes_))
print(encoder.classes_)

Number of classes: 41
['(vertigo) Paroymsal  Positional Vertigo' 'AIDS' 'Acne'
 'Alcoholic hepatitis' 'Allergy' 'Arthritis' 'Bronchial Asthma'
 'Cervical spondylosis' 'Chicken pox' 'Chronic cholestasis' 'Common Cold'
 'Dengue' 'Diabetes ' 'Dimorphic hemmorhoids(piles)' 'Drug Reaction'
 'Fungal infection' 'GERD' 'Gastroenteritis' 'Heart attack' 'Hepatitis B'
 'Hepatitis C' 'Hepatitis D' 'Hepatitis E' 'Hypertension '
 'Hyperthyroidism' 'Hypoglycemia' 'Hypothyroidism' 'Impetigo' 'Jaundice'
 'Malaria' 'Migraine' 'Osteoarthristis' 'Paralysis (brain hemorrhage)'
 'Peptic ulcer diseae' 'Pneumonia' 'Psoriasis' 'Tuberculosis' 'Typhoid'
 'Urinary tract infection' 'Varicose veins' 'hepatitis A']


In [0]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [0]:
print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (243, 132)
Testing : (61, 132)


In [0]:
print("Training:", X_train.shape)
print("Testing :", X_test.shape)

print("\nNumber of disease classes:", len(encoder.classes_))

print("\nDisease distribution:")
print(train_clean["prognosis"].value_counts().sort_index())

Training: (243, 132)
Testing : (61, 132)

Number of disease classes: 41

Disease distribution:
prognosis
(vertigo) Paroymsal  Positional Vertigo     7
AIDS                                        5
Acne                                        5
Alcoholic hepatitis                         8
Allergy                                     5
Arthritis                                   6
Bronchial Asthma                            7
Cervical spondylosis                        6
Chicken pox                                10
Chronic cholestasis                         8
Common Cold                                 9
Dengue                                     10
Diabetes                                    9
Dimorphic hemmorhoids(piles)                6
Drug Reaction                               6
Fungal infection                            5
GERD                                        7
Gastroenteritis                             5
Heart attack                                5
Hepatitis B          

In [0]:
import pandas as pd

train_distribution = pd.Series(y_train).value_counts().sort_index()
test_distribution = pd.Series(y_test).value_counts().sort_index()

comparison = pd.DataFrame({
    "Train": train_distribution,
    "Test": test_distribution
}).fillna(0)

display(comparison)

Train,Test
6,1
4,1
4,1
6,2
4,1
5,1
6,1
5,1
8,2
6,2


Verify No NaN

In [0]:
print("Training NaN:", X_train.isna().sum().sum())
print("Testing NaN :", X_test.isna().sum().sum())

Training NaN: 0
Testing NaN : 0


### Decision Tree

In [0]:
dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [0]:
y_pred_dt = dt_model.predict(X_test)

Metrics

In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

dt_accuracy = accuracy_score(y_test, y_pred_dt)

dt_precision = precision_score(
    y_test,
    y_pred_dt,
    average="weighted",
    zero_division=0
)

dt_recall = recall_score(
    y_test,
    y_pred_dt,
    average="weighted",
    zero_division=0
)

dt_f1 = f1_score(
    y_test,
    y_pred_dt,
    average="weighted",
    zero_division=0
)

print("Decision Tree")
print("Accuracy :", dt_accuracy)
print("Precision:", dt_precision)
print("Recall   :", dt_recall)
print("F1 Score :", dt_f1)

Decision Tree
Accuracy : 0.7049180327868853
Precision: 0.7124316939890709
Recall   : 0.7049180327868853
F1 Score : 0.6812386156648452


/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99

### Naive Bayes

In [0]:
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

,priors,None
,var_smoothing,1e-09


Prediction

In [0]:
y_pred_nb = nb_model.predict(X_test)

Metrics:

In [0]:
nb_accuracy = accuracy_score(y_test, y_pred_nb)

nb_precision = precision_score(
    y_test,
    y_pred_nb,
    average="weighted",
    zero_division=0
)

nb_recall = recall_score(
    y_test,
    y_pred_nb,
    average="weighted",
    zero_division=0
)

nb_f1 = f1_score(
    y_test,
    y_pred_nb,
    average="weighted",
    zero_division=0
)

print("Naive Bayes")
print("Accuracy :", nb_accuracy)
print("Precision:", nb_precision)
print("Recall   :", nb_recall)
print("F1 Score :", nb_f1)

Naive Bayes
Accuracy : 0.9836065573770492
Precision: 0.989071038251366
Recall   : 0.9836065573770492
F1 Score : 0.9825136612021859


/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99

### Random Forest

In [0]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


Prediction:

In [0]:
y_pred_rf = rf_model.predict(X_test)

Metrics

In [0]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)

rf_precision = precision_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

rf_recall = recall_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

rf_f1 = f1_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

print("Random Forest")
print("Accuracy :", rf_accuracy)
print("Precision:", rf_precision)
print("Recall   :", rf_recall)
print("F1 Score :", rf_f1)

Random Forest
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99

### Compare All Three Models

In [0]:
results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Naive Bayes",
        "Random Forest"
    ],

    "Accuracy": [
        dt_accuracy,
        nb_accuracy,
        rf_accuracy
    ],

    "Precision": [
        dt_precision,
        nb_precision,
        rf_precision
    ],

    "Recall": [
        dt_recall,
        nb_recall,
        rf_recall
    ],

    "F1 Score": [
        dt_f1,
        nb_f1,
        rf_f1
    ]
})

display(results)

Model,Accuracy,Precision,Recall,F1 Score
Decision Tree,0.7049180327868853,0.7124316939890709,0.7049180327868853,0.6812386156648452
Naive Bayes,0.9836065573770492,0.989071038251366,0.9836065573770492,0.9825136612021859
Random Forest,1.0,1.0,1.0,1.0


### Show Percentages

In [0]:
results_percentage = results.copy()

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
]

results_percentage[metrics] = (
    results_percentage[metrics] * 100
).round(2)

display(results_percentage)

Model,Accuracy,Precision,Recall,F1 Score
Decision Tree,70.49,71.24,70.49,68.12
Naive Bayes,98.36,98.91,98.36,98.25
Random Forest,100.0,100.0,100.0,100.0


### Check Training vs Testing Accuracy

In [0]:
dt_train_acc = accuracy_score(
    y_train,
    dt_model.predict(X_train)
)

dt_test_acc = accuracy_score(
    y_test,
    y_pred_dt
)

rf_train_acc = accuracy_score(
    y_train,
    rf_model.predict(X_train)
)

rf_test_acc = accuracy_score(
    y_test,
    y_pred_rf
)

print("Decision Tree")
print("Train:", dt_train_acc)
print("Test :", dt_test_acc)

print("\nRandom Forest")
print("Train:", rf_train_acc)
print("Test :", rf_test_acc)

Decision Tree
Train: 1.0
Test : 0.7049180327868853

Random Forest
Train: 1.0
Test : 1.0


/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99

First check the minimum records per disease

In [0]:
class_counts = train_clean["prognosis"].value_counts()

print("Minimum records for any disease:",
      class_counts.min())

print("\nDisease distribution:")
display(class_counts.sort_values().to_frame("count"))

Minimum records for any disease: 5

Disease distribution:


count
5
5
5
5
5
5
5
5
6
6


Cross-validation for Random Forest

In [0]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier

rf_cv_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted"
}

rf_cv_results = cross_validate(
    rf_cv_model,
    X,
    y_encoded,
    cv=cv,
    scoring=scoring
)

/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99

Now calculate the averages:

In [0]:
print(
    "Random Forest Cross-Validation Results"
)

print(
    "Accuracy :",
    rf_cv_results["test_accuracy"].mean()
)

print(
    "Precision:",
    rf_cv_results["test_precision"].mean()
)

print(
    "Recall   :",
    rf_cv_results["test_recall"].mean()
)

print(
    "F1 Score :",
    rf_cv_results["test_f1"].mean()
)

Random Forest Cross-Validation Results
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


Do the same for Decision Tree

In [0]:
dt_cv_model = DecisionTreeClassifier(
    random_state=42
)

dt_cv_results = cross_validate(
    dt_cv_model,
    X,
    y_encoded,
    cv=cv,
    scoring=scoring
)

/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: Th

In [0]:
print(
    "Decision Tree Cross-Validation Results"
)

print(
    "Accuracy :",
    dt_cv_results["test_accuracy"].mean()
)

print(
    "Precision:",
    dt_cv_results["test_precision"].mean()
)

print(
    "Recall   :",
    dt_cv_results["test_recall"].mean()
)

print(
    "F1 Score :",
    dt_cv_results["test_f1"].mean()
)

Decision Tree Cross-Validation Results
Accuracy : 0.6447368421052632
Precision: 0.7568202356853672
Recall   : 0.6447368421052632
F1 Score : 0.6636215301511353


Naive Bayes Cross-Validation

In [0]:
from sklearn.naive_bayes import GaussianNB

nb_cv_model = GaussianNB()

nb_cv_results = cross_validate(
    nb_cv_model,
    X,
    y_encoded,
    cv=cv,
    scoring=scoring
)

print(
    "Naive Bayes Cross-Validation Results"
)

print(
    "Accuracy :",
    nb_cv_results["test_accuracy"].mean()
)

print(
    "Precision:",
    nb_cv_results["test_precision"].mean()
)

print(
    "Recall   :",
    nb_cv_results["test_recall"].mean()
)

print(
    "F1 Score :",
    nb_cv_results["test_f1"].mean()
)

Naive Bayes Cross-Validation Results
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99

Create the Final Cross-Validation Comparison

In [0]:
cv_results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Naive Bayes",
        "Random Forest"
    ],

    "Accuracy": [
        dt_cv_results["test_accuracy"].mean(),
        nb_cv_results["test_accuracy"].mean(),
        rf_cv_results["test_accuracy"].mean()
    ],

    "Precision": [
        dt_cv_results["test_precision"].mean(),
        nb_cv_results["test_precision"].mean(),
        rf_cv_results["test_precision"].mean()
    ],

    "Recall": [
        dt_cv_results["test_recall"].mean(),
        nb_cv_results["test_recall"].mean(),
        rf_cv_results["test_recall"].mean()
    ],

    "F1 Score": [
        dt_cv_results["test_f1"].mean(),
        nb_cv_results["test_f1"].mean(),
        rf_cv_results["test_f1"].mean()
    ]
})

display(cv_results)

Model,Accuracy,Precision,Recall,F1 Score
Decision Tree,0.6447368421052632,0.7568202356853672,0.6447368421052632,0.6636215301511353
Naive Bayes,1.0,1.0,1.0,1.0
Random Forest,1.0,1.0,1.0,1.0


In [0]:
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
]

cv_results_percentage = cv_results.copy()

cv_results_percentage[metrics] = (
    cv_results_percentage[metrics] * 100
).round(2)

display(cv_results_percentage)

Model,Accuracy,Precision,Recall,F1 Score
Decision Tree,64.47,75.68,64.47,66.36
Naive Bayes,100.0,100.0,100.0,100.0
Random Forest,100.0,100.0,100.0,100.0


Run the first check now:

In [0]:
class_counts = train_clean["prognosis"].value_counts()

print("Minimum records for any disease:",
      class_counts.min())

display(class_counts.sort_values().to_frame("count"))

Minimum records for any disease: 5


count
5
5
5
5
5
5
5
5
6
6


"Duplicate symptom patterns:

In [0]:
print("Total unique records:", len(train_clean))

print(
    "Duplicate symptom patterns:",
    X.duplicated().sum()
)

Total unique records: 304
Duplicate symptom patterns: 0


check whether each disease has a unique symptom pattern

In [0]:
pattern_check = (
    train_clean
    .groupby(list(X.columns))["prognosis"]
    .nunique()
)

print(
    "Conflicting symptom patterns:",
    (pattern_check > 1).sum()
)

Conflicting symptom patterns: 0


### Disease Prediction System

In [0]:
model_path = "/Volumes/workspace/disease_prediction/models/final_disease_prediction_model.pkl"
encoder_path = "/Volumes/workspace/disease_prediction/models/disease_label_encoder.pkl"
feature_path = "/Volumes/workspace/disease_prediction/models/feature_columns.pkl"

rf_model = joblib.load(model_path)
encoder = joblib.load(encoder_path)
feature_columns = joblib.load(feature_path)

print("Model loaded")
print("Encoder loaded")
print("Number of symptoms:", len(feature_columns))

Model loaded
Encoder loaded
Number of symptoms: 132


### Create predict_disease()

In [0]:
def predict_disease(symptoms):

    # Create a row with all symptoms set to 0
    patient = pd.DataFrame(
        0,
        index=[0],
        columns=feature_columns
    )

    # Set selected symptoms to 1
    for symptom in symptoms:
        if symptom in patient.columns:
            patient.loc[0, symptom] = 1
        else:
            print(f"Warning: '{symptom}' is not a valid symptom")

    # Predict disease
    prediction = rf_model.predict(patient)

    # Convert encoded number back to disease name
    predicted_disease = encoder.inverse_transform(prediction)

    return predicted_disease[0]

In [0]:
disease = predict_disease([
    "itching",
    "skin_rash",
    "fatigue"
])

print("Predicted Disease:", disease)

Predicted Disease: Fungal infection


# Databricks Disease Prediction Dashboard ---------

Create a prediction table

In [0]:
prediction_df = X_test.copy()

prediction_df["actual_disease"] = encoder.inverse_transform(y_test)

prediction_df["predicted_disease"] = encoder.inverse_transform(
    rf_model.predict(X_test)
)

prediction_df["correct"] = (
    prediction_df["actual_disease"]
    == prediction_df["predicted_disease"]
)

Check it:

In [0]:
display(prediction_df.head())

itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,muscle_wasting,vomiting,burning_micturition,spotting_ urination,fatigue,weight_gain,anxiety,cold_hands_and_feets,mood_swings,weight_loss,restlessness,lethargy,patches_in_throat,irregular_sugar_level,cough,high_fever,sunken_eyes,breathlessness,sweating,dehydration,indigestion,headache,yellowish_skin,dark_urine,nausea,loss_of_appetite,pain_behind_the_eyes,back_pain,constipation,abdominal_pain,diarrhoea,mild_fever,yellow_urine,yellowing_of_eyes,acute_liver_failure,fluid_overload,swelling_of_stomach,swelled_lymph_nodes,malaise,blurred_and_distorted_vision,phlegm,throat_irritation,redness_of_eyes,sinus_pressure,runny_nose,congestion,chest_pain,weakness_in_limbs,fast_heart_rate,pain_during_bowel_movements,pain_in_anal_region,bloody_stool,irritation_in_anus,neck_pain,dizziness,cramps,bruising,obesity,swollen_legs,swollen_blood_vessels,puffy_face_and_eyes,enlarged_thyroid,brittle_nails,swollen_extremeties,excessive_hunger,extra_marital_contacts,drying_and_tingling_lips,slurred_speech,knee_pain,hip_joint_pain,muscle_weakness,stiff_neck,swelling_joints,movement_stiffness,spinning_movements,loss_of_balance,unsteadiness,weakness_of_one_body_side,loss_of_smell,bladder_discomfort,foul_smell_of urine,continuous_feel_of_urine,passage_of_gases,internal_itching,toxic_look_(typhos),depression,irritability,muscle_pain,altered_sensorium,red_spots_over_body,belly_pain,abnormal_menstruation,dischromic _patches,watering_from_eyes,increased_appetite,polyuria,family_history,mucoid_sputum,rusty_sputum,lack_of_concentration,visual_disturbances,receiving_blood_transfusion,receiving_unsterile_injections,coma,stomach_bleeding,distention_of_abdomen,history_of_alcohol_consumption,fluid_overload.1,blood_in_sputum,prominent_veins_on_calf,palpitations,painful_walking,pus_filled_pimples,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,actual_disease,predicted_disease,correct
1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Hepatitis B,Hepatitis B,true
0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Pneumonia,Pneumonia,true
1,1,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Drug Reaction,Drug Reaction,true
0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Allergy,Allergy,true
0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Pneumonia,Pneumonia,true


#### Save it as a Unity Catalog table

In [0]:
prediction_df_clean = prediction_df.copy()

prediction_df_clean.columns = (
    prediction_df_clean.columns
    .str.replace(" ", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

spark_prediction_df = spark.createDataFrame(prediction_df_clean)

spark_prediction_df.write.mode("overwrite").saveAsTable(
    "workspace.disease_prediction.predictions"
)

Then verify:

In [0]:
display(
    spark.table("workspace.disease_prediction.predictions")
)

itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,muscle_wasting,vomiting,burning_micturition,spotting__urination,fatigue,weight_gain,anxiety,cold_hands_and_feets,mood_swings,weight_loss,restlessness,lethargy,patches_in_throat,irregular_sugar_level,cough,high_fever,sunken_eyes,breathlessness,sweating,dehydration,indigestion,headache,yellowish_skin,dark_urine,nausea,loss_of_appetite,pain_behind_the_eyes,back_pain,constipation,abdominal_pain,diarrhoea,mild_fever,yellow_urine,yellowing_of_eyes,acute_liver_failure,fluid_overload,swelling_of_stomach,swelled_lymph_nodes,malaise,blurred_and_distorted_vision,phlegm,throat_irritation,redness_of_eyes,sinus_pressure,runny_nose,congestion,chest_pain,weakness_in_limbs,fast_heart_rate,pain_during_bowel_movements,pain_in_anal_region,bloody_stool,irritation_in_anus,neck_pain,dizziness,cramps,bruising,obesity,swollen_legs,swollen_blood_vessels,puffy_face_and_eyes,enlarged_thyroid,brittle_nails,swollen_extremeties,excessive_hunger,extra_marital_contacts,drying_and_tingling_lips,slurred_speech,knee_pain,hip_joint_pain,muscle_weakness,stiff_neck,swelling_joints,movement_stiffness,spinning_movements,loss_of_balance,unsteadiness,weakness_of_one_body_side,loss_of_smell,bladder_discomfort,foul_smell_of_urine,continuous_feel_of_urine,passage_of_gases,internal_itching,toxic_look_typhos,depression,irritability,muscle_pain,altered_sensorium,red_spots_over_body,belly_pain,abnormal_menstruation,dischromic__patches,watering_from_eyes,increased_appetite,polyuria,family_history,mucoid_sputum,rusty_sputum,lack_of_concentration,visual_disturbances,receiving_blood_transfusion,receiving_unsterile_injections,coma,stomach_bleeding,distention_of_abdomen,history_of_alcohol_consumption,fluid_overload.1,blood_in_sputum,prominent_veins_on_calf,palpitations,painful_walking,pus_filled_pimples,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,actual_disease,predicted_disease,correct
1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Hepatitis B,Hepatitis B,true
0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Pneumonia,Pneumonia,true
1,1,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Drug Reaction,Drug Reaction,true
0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Allergy,Allergy,true
0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Pneumonia,Pneumonia,true
0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Hypothyroidism,Hypothyroidism,true
0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

Create a Model Comparison table

In [0]:
cv_results_clean = cv_results.copy()

cv_results_clean.columns = (
    cv_results_clean.columns
    .str.replace(" ", "_", regex=False)
)

spark_cv_results = spark.createDataFrame(cv_results_clean)

spark_cv_results.write.mode("overwrite").saveAsTable(
    "workspace.disease_prediction.model_comparison"
)

Check:

In [0]:
display(
    spark.table(
        "workspace.disease_prediction.model_comparison"
    )
)

Model,Accuracy,Precision,Recall,F1_Score
Decision Tree,0.6447368421052632,0.7568202356853672,0.6447368421052632,0.6636215301511353
Naive Bayes,1.0,1.0,1.0,1.0
Random Forest,1.0,1.0,1.0,1.0


#### Save Random Forest Feature Importance

In [0]:
import pandas as pd

# Calculate feature importance from the trained Random Forest model
importance = rf_model.feature_importances_

# Create the DataFrame
importance_df = pd.DataFrame({
    "Symptom": X_train.columns,
    "Importance": importance
})

# Sort by importance
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

display(importance_df.head(20))

Symptom,Importance
itching,0.021057912305987497
muscle_pain,0.020126629910746575
fatigue,0.019954764871765097
vomiting,0.01906103838879368
joint_pain,0.018298936128287775
yellowing_of_eyes,0.018039317539122097
chills,0.016750894090791016
high_fever,0.0164375395316902
mild_fever,0.016225339246021833
nausea,0.01569455699581308


In [0]:

spark_importance = spark.createDataFrame(
    importance_df
)

spark_importance.write.mode("overwrite").saveAsTable(
    "workspace.disease_prediction.feature_importance"
)

In [0]:
%sql
SELECT Symptom, Importance
FROM workspace.disease_prediction.feature_importance
ORDER BY Importance DESC
LIMIT 20;

Symptom,Importance
itching,0.021057912305987497
muscle_pain,0.020126629910746575
fatigue,0.019954764871765097
vomiting,0.01906103838879368
joint_pain,0.018298936128287775
yellowing_of_eyes,0.018039317539122097
chills,0.016750894090791016
high_fever,0.0164375395316902
mild_fever,0.016225339246021833
nausea,0.01569455699581308


# Create the Dashboard

In [0]:
import pandas as pd

# Generate predictions on the test dataset
y_pred = rf_model.predict(X_test)

# Create results DataFrame
prediction_df = X_test.copy()

prediction_df["actual_disease"] = encoder.inverse_transform(y_test)
prediction_df["predicted_disease"] = encoder.inverse_transform(y_pred)

prediction_df["correct"] = (
    prediction_df["actual_disease"] ==
    prediction_df["predicted_disease"]
)

# Sanitize column names (spaces/parens are invalid in UC Delta columns)
prediction_df.columns = (
    prediction_df.columns
    .str.replace(" ", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

# Save as Unity Catalog table
spark_prediction_df = spark.createDataFrame(prediction_df)

spark_prediction_df.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.disease_prediction.predictions")

print("Predictions table created!")




Predictions table created!


Create the Feature Importance table

In [0]:
importance_df = pd.DataFrame({
    "Symptom": X_train.columns,
    "Importance": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

spark.createDataFrame(importance_df) \
    .write.mode("overwrite") \
    .saveAsTable("workspace.disease_prediction.feature_importance")

print("Feature importance table created!")

Feature importance table created!


Create the Model Comparison table

In [0]:
model_comparison_df = pd.DataFrame([
    {
        "Model": "Decision Tree",
        "Accuracy": 64.47,
        "Precision": 75.68,
        "Recall": 64.47,
        "F1_Score": 66.36
    },
    {
        "Model": "Naive Bayes",
        "Accuracy": 100.00,
        "Precision": 100.00,
        "Recall": 100.00,
        "F1_Score": 100.00
    },
    {
        "Model": "Random Forest",
        "Accuracy": 100.00,
        "Precision": 100.00,
        "Recall": 100.00,
        "F1_Score": 100.00
    }
])

spark.createDataFrame(model_comparison_df) \
    .write.mode("overwrite") \
    .saveAsTable("workspace.disease_prediction.model_comparison")

print("Model comparison table created!")model_comparison_df = pd.DataFrame([
    {
        "Model": "Decision Tree",
        "Accuracy": 64.47,
        "Precision": 75.68,
        "Recall": 64.47,
        "F1_Score": 66.36
    },
    {
        "Model": "Naive Bayes",
        "Accuracy": 100.00,
        "Precision": 100.00,
        "Recall": 100.00,
        "F1_Score": 100.00
    },
    {
        "Model": "Random Forest",
        "Accuracy": 100.00,
        "Precision": 100.00,
        "Recall": 100.00,
        "F1_Score": 100.00
    }
])

spark.createDataFrame(model_comparison_df) \
    .write.mode("overwrite") \
    .saveAsTable("workspace.disease_prediction.model_comparison")

print("Model comparison table created!")

Create the Data Quality table

In [0]:
# Use the original and cleaned Pandas DataFrames
original_rows = len(train)
cleaned_rows = len(train_clean)

# Exclude the target column to count symptom features
symptom_columns = [
    col for col in train_clean.columns
    if col != "prognosis"
]

quality_df = pd.DataFrame([{
    "Original_Rows": original_rows,
    "Cleaned_Rows": cleaned_rows,
    "Duplicates_Removed": original_rows - cleaned_rows,
    "Symptom_Features": len(symptom_columns),
    "Disease_Classes": train_clean["prognosis"].nunique(),
    "Missing_Values": int(train_clean.isnull().sum().sum())
}])

spark.createDataFrame(quality_df) \
    .write.mode("overwrite") \
    .saveAsTable("workspace.disease_prediction.data_quality")

display(quality_df)

Original_Rows,Cleaned_Rows,Duplicates_Removed,Symptom_Features,Disease_Classes,Missing_Values
4920,304,4616,132,41,0


In [0]:
%sql
SHOW TABLES IN workspace.disease_prediction

database,tableName,isTemporary
disease_prediction,data_quality,false
disease_prediction,feature_importance,false
disease_prediction,model_comparison,false
disease_prediction,predictions,false
